In [ ]:
!pip install -q kaggle

from google.colab import files
print("Upload your kaggle.json file")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Upload your kaggle.json file


Saving kaggle.json to kaggle.json


In [ ]:
!kaggle datasets download -d snmahsa/animal-image-dataset-cats-dogs-and-foxes
!unzip -q animal-image-dataset-cats-dogs-and-foxes.zip -d dataset
!find dataset -maxdepth 3 -type d

Dataset URL: https://www.kaggle.com/datasets/snmahsa/animal-image-dataset-cats-dogs-and-foxes
License(s): other
100% 388M/388M [00:03<00:00, 125MB/s]

dataset
dataset/Animal Image Dataset-Cats, Dogs, and Foxes
dataset/Animal Image Dataset-Cats, Dogs, and Foxes/fox
dataset/Animal Image Dataset-Cats, Dogs, and Foxes/cat
dataset/Animal Image Dataset-Cats, Dogs, and Foxes/dog


In [ ]:
import os, shutil, glob, random

SRC = "dataset"
BASE = "data"
random.seed(42)

def find_class_dir(root, name):
    for dirpath, dirnames, _ in os.walk(root):
        for d in dirnames:
            if d.lower() == name:
                return os.path.join(dirpath, d)
    return None

cats_dir = find_class_dir(SRC, "cats") or find_class_dir(SRC, "cat")
foxes_dir = find_class_dir(SRC, "foxes") or find_class_dir(SRC, "fox")
print("cats:", cats_dir)
print("foxes:", foxes_dir)

for split in ["train", "val"]:
    for cls in ["cats", "foxes"]:
        os.makedirs(f"{BASE}/{split}/{cls}", exist_ok=True)

def split_and_copy(src_dir, cls, val_ratio=0.2):
    imgs = glob.glob(os.path.join(src_dir, "*"))
    random.shuffle(imgs)
    n_val = int(len(imgs) * val_ratio)
    val_imgs = imgs[:n_val]
    train_imgs = imgs[n_val:]
    for f in train_imgs:
        shutil.copy(f, f"{BASE}/train/{cls}/")
    for f in val_imgs:
        shutil.copy(f, f"{BASE}/val/{cls}/")
    print(f"{cls}: {len(train_imgs)} train, {len(val_imgs)} val")

split_and_copy(cats_dir, "cats")
split_and_copy(foxes_dir, "foxes")

cats: dataset/Animal Image Dataset-Cats, Dogs, and Foxes/cat
foxes: dataset/Animal Image Dataset-Cats, Dogs, and Foxes/fox
cats: 83 train, 20 val
foxes: 82 train, 20 val


In [ ]:
import tensorflow as tf

print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

if tf.test.is_built_with_cuda():
    print("TensorFlow is built with CUDA (GPU support).")
else:
    print("TensorFlow is NOT built with CUDA. Running on CPU or other device.")

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        print(e)

Num GPUs Available:  1
TensorFlow is built with CUDA (GPU support).
1 Physical GPUs, 1 Logical GPUs


In [ ]:
from tensorflow.keras import layers, models

IMG_SIZE = (160, 160)
BATCH_SIZE = 16

train_ds = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "data/val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

class_names = train_ds.class_names
print("Classes:", class_names)

Found 164 files belonging to 2 classes.
Found 39 files belonging to 2 classes.
Classes: ['cats', 'foxes']


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.2),
    ]
)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
EPOCHS = 15

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Epoch 1/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 39s 2s/step - accuracy: 0.4939 - loss: 0.7642 - val_accuracy: 0.4615 - val_loss: 0.7677
Epoch 2/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 33s 3s/step - accuracy: 0.6159 - loss: 0.6875 - val_accuracy: 0.5128 - val_loss: 0.7259
Epoch 3/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 37s 3s/step - accuracy: 0.5732 - loss: 0.6874 - val_accuracy: 0.5385 - val_loss: 0.6838
Epoch 4/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 32s 3s/step - accuracy: 0.6159 - loss: 0.6869 - val_accuracy: 0.6154 - val_loss: 0.6434
Epoch 5/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 37s 3s/step - accuracy: 0.7012 - loss: 0.5907 - val_accuracy: 0.6410 - val_loss: 0.6075
Epoch 6/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 41s 3s/step - accuracy: 0.7256 - loss: 0.5832 - val_accuracy: 0.6667 - val_loss: 0.5762
Epoch 7/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 28s 3s/step - accuracy: 0.7622 - loss: 0.5199 - val_accuracy: 0.6923 - val_loss: 0.5495
Epoch 8/15
11/11 ━━━━━━━━━━━━━━━━━━━━ 41s 3s/step - accuracy: 0.7561 - loss: 0.5109 - val_accuracy: 0.7436 - val_loss:

In [ ]:
model.save("cat_fox_classifier.keras")

from google.colab import files
files.download("cat_fox_classifier.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>